In [100]:
### RAG pipeline - Data Ingestion to Vector DB pipline

In [101]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [102]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

def process_all_txts(txt_directory):
    """Process all .txt files in a directory"""
    all_documents = []
    txt_dir = Path(txt_directory)

    # Find all .txt files recursively
    txt_files = list(txt_dir.glob("**/*.txt"))

    print(f"Found {len(txt_files)} txt files to process")

    for txt_file in txt_files:
        print(f"\nProcessing: {txt_file.name}")
        try:
            loader = TextLoader(str(txt_file), encoding="utf-8", autodetect_encoding=True)
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = txt_file.name
                doc.metadata['file_type'] = 'txt'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} document(s)")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all .txt files in the data directory
all_txt_documents = process_all_txts("../data")

Found 1 txt files to process

Processing: triple_axis_scan.txt
  ✓ Loaded 1 document(s)

Total documents loaded: 1


In [103]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_documents = process_all_pdfs("../data") + process_all_txts("../data")

Found 8 PDF files to process

Processing: Staica 1975 Part1.pdf
  ✓ Loaded 4 pages

Processing: Popvici 1975 part4.pdf
  ✓ Loaded 7 pages

Processing: Cooper-Nathans original 1967.pdf
  ✓ Loaded 11 pages

Processing: Popvici and stoica 1975 part3.pdf
  ✓ Loaded 4 pages

Processing: Stoica 1975 part2.pdf
  ✓ Loaded 4 pages

Processing: reslib.pdf
  ✓ Loaded 47 pages

Processing: Werner and Pynn 1971.pdf
  ✓ Loaded 15 pages

Processing: Mitchell - 1984.pdf
  ✓ Loaded 9 pages

Total documents loaded: 101
Found 1 txt files to process

Processing: triple_axis_scan.txt
  ✓ Loaded 1 document(s)

Total documents loaded: 1


In [104]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
chunks=split_documents(all_documents)

Split 102 documents into 386 chunks

Example chunk:
Content: 189 
Acta Cryst. (1975). A31, 189 
On the Resolution of Slow-Neutron Spectrometers. 
I. A General Method to Calculate Resolution Functions 
By A. D. STOICA 
Institute for Atomic Physics, Bucharest, P....
Metadata: {'producer': 'PageGenie PDFGenerator; modified using iText 4.2.0 by 1T3XT', 'creator': 'a11676.pdf', 'creationdate': 'Wed May 16 23:41:23 2001', 'keywords': '', 'moddate': '2026-04-06T09:46:25-07:00', 'subject': 'Acta Crystallographica Section A 1975.31:189-192', 'author': '', 'title': 'On the resolution of slow‐neutron spectrometers. I. A general method to calculate resolution functions', 'source': '../data/pdf/Staica 1975 Part1.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Staica 1975 Part1.pdf', 'file_type': 'pdf'}


In [105]:
### embedding and vectorStoreDB

In [106]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity



In [107]:
class EmbedingManager:
    """Hanldes document embedding generation using SentenceTransformer."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize embedding manager."""
        self.model_name = model_name
        self.model = None
        self._load_model() 
    
    def _load_model(self):
        try:
            print(f"Loading embeding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]):
        if not self.model:
            raise ValueError("Model not loaded.")
        print(f"generating embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings

embedding_manager = EmbedingManager()
embedding_manager

Loading embeding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11617.32it/s]


Model loaded successfully. Embedding dimension: 384


In [108]:
### Vector Store

class VectorStore:
    """Manage document embeddings in a chromaDB vector store."""
    def __init__(self, collection_name: str = "pdf_documents", persistent_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persistent_directory, exist_ok =True)
            self.client = chromadb.PersistentClient(path = self.persistent_directory)
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialied. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            raise ValueError(e)
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store....")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
        except Exception as e:
            raise ValueError("e")
vectorstore = VectorStore()

Vector store initialied. Collection: pdf_documents
Existing documents in collection: 1157


In [109]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

generating embedding for 386 texts...


Batches: 100%|██████████| 13/13 [00:00<00:00, 15.25it/s]


Generated embeddings with shape : (386, 384)
Adding 386 documents to vector store....


In [110]:
### Retriever pipeline from VectorStore

class RAGRetriever:
    """Handles query-based retrieval from the vector store."""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbedingManager):
        """Init."""
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> list[Dict[str, Any]]:
        """Retrieve relevant documents for a query."""
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found.")
            return retrieved_docs
        except Exception as e:
            print(e)  
            return []

rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager= embedding_manager)

In [111]:
### simple rag pipline
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(model="gemma4:latest", temperature=0.1, max_completion_tokens=1024)

def rag_simple(query, retriever, llm, top_k=3):
    # retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # generate the answer
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke([
    SystemMessage("Answer concisely using only the provided context."),
    HumanMessage(f"Context:\n{context}\n\nQuestion: {query}"),
])
    return response.content


In [112]:
answer = rag_simple("Generatea a scan command of h from 0 to 1 with a step size of 0.1", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'Generatea a scan command of h from 0 to 1 with a step size of 0.1'
Top K: 3, score threshold: 0.0
generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 128.98it/s]

Generated embeddings with shape : (1, 384)
Retrieved 2 documents (after filtering)


scan h 0 1 0.1 k 0 l 0 e 0


In [113]:
### Generate triple-axis scan commands from a natural-language request

SCAN_SYSTEM_PROMPT = """You are an assistant that converts natural-language requests \
into Triple Axis instrument scan commands.

Rules:
- Use ONLY the command syntax shown in the provided context.
- Output ONLY the command(s), one per line. No explanation, no markdown, no code fences.
- If a monitor count (preset) or scan title is requested, emit those commands first.
- For a reciprocal-space scan use: scan h <start> <stop> <step> k <k> l <l> e <e>
- If the request is missing a value, use 0 for that coordinate.
- If the request cannot be expressed with the documented commands, reply exactly: UNSUPPORTED"""


def generate_scan_command(request, retriever, llm, top_k=3):
    """Translate a natural-language request into triple-axis scan command(s)."""
    # Retrieve the command syntax from the knowledge base
    results = retriever.retrieve(request, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No command syntax found in the knowledge base."

    response = llm.invoke([
        SystemMessage(SCAN_SYSTEM_PROMPT),
        HumanMessage(f"Command reference:\n{context}\n\nRequest: {request}\n\nCommands:"),
    ])
    return response.content.strip()


In [ ]:
bragg_peaks = ([0, 0, 1], [1, 2, 1], [3, 2, 1])
# Example
cmd = generate_scan_command(
    f"move to the following Bragg peak positions {bragg_peaks}, and generate a command with appropriate titles of scanning h plus minus 0.5, step size 0.1, preset mcu 100 for each Bragg peak",
    rag_retriever,
    llm,
)
print(cmd)

Retrieving documents for query: 'move to the following Bragg peak positions ([0, 0, 1], [1, 2, 1], [3, 2, 1]), and generate a series of command with appropriate titles of scanning h plus minus 0.5, step size 0.1, preset mcu 100'
Top K: 3, score threshold: 0.0
generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.95it/s]

Generated embeddings with shape : (1, 384)
Retrieved 3 documents (after filtering)


preset mcu 100
scantitle "h +/- 0.5"
br 0 0 1
br 1 2 1
br 3 2 1
scan h 0.5 1.5 0.1 k 0 l 0 e 0


In [115]:
bragg_peaks = ([0, 0, 1], [1, 2, 1], [3, 2, 1])
# Example
cmd = generate_scan_command(
    f"move to the following Bragg peak positions {bragg_peaks}, and generate a series of command with appropriate titles of inelastic scans from 0 to 10 meV searching for spin gaps, preset mcu 100",
    rag_retriever,
    llm,
)
print(cmd)

Retrieving documents for query: 'move to the following Bragg peak positions ([0, 0, 1], [1, 2, 1], [3, 2, 1]), and generate a series of command with appropriate titles of inelastic scans from 0 to 10 meV searching for spin gaps, preset mcu 100'
Top K: 3, score threshold: 0.0
generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.93it/s]

Generated embeddings with shape : (1, 384)
Retrieved 1 documents (after filtering)


preset mcu 100
scantitle "Search for spin gaps"
br 0 0 1
br 1 2 1
br 3 2 1
scan h 0 k 0 l 0 e 0 10 0.1
